# `polartox.datagen` — Synthetic Data Generation Demo

Polarized Trees needs datasets where the ground truth is *known* — which demographic dimensions drive disagreement, and how strongly. Real annotation data can't provide this, so we generate synthetic datasets with **injected, controlled polarization**.

Each text independently gets:
- **k active dimensions** (0–4), drawn per text — `k=0` is a true unimodal **negative control**, no dimension explains anything
- for each active dimension, a random **toxic/civil lean split** across its values
- one **intensity** `alpha ∈ [alpha_min, alpha_max]` per active dimension, controlling how strongly it pulls toward its pole

Each identity's rating distribution is the **elementwise product** of its active-dimension shapes — signal composes instead of averaging away, so the full nDFU range (0 to ~0.9) is reachable, unlike naive weight-averaging approaches.

This notebook walks through every public method on `AnnotatorPool` and `GeneratedDataset`, in order.

**Requires:** `pip install polartox`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from polartox.datagen import AnnotatorPool, DEFAULT_DIMENSIONS, DEFAULT_DEPTH_WEIGHTS, DEFAULT_INTENSITY_RANGE

## 1. Build the annotator pool

`DEFAULT_DIMENSIONS` covers gender, politics, age, education, and orientation (162 identity cells).
Pass a custom dict to use different dimensions or values.
All parameters are mandatory — nothing is taken for granted implicitly. If you want the reference configuration, pass the `DEFAULT_*` constants explicitly, as below.

In [ ]:
pool = AnnotatorPool(
    dimensions=DEFAULT_DIMENSIONS,
    scale=5,                              # rating scale is 1..scale
    intensity_range=DEFAULT_INTENSITY_RANGE,  # (alpha_min, alpha_max), each active dim draws its own alpha
    depth_weights=DEFAULT_DEPTH_WEIGHTS,      # P(k active dims) for k = 0..4
    annotators_per_identity=10,           # annotators replicated per unique identity
)
pool.summary()

## 2. Generate a dataset

Every text is an **independent draw**: how many dimensions are active (`k`), which ones, their lean split, and each one's intensity `alpha` are all generated fresh per text. By default every pool annotator rates every text — pass `n_annotators_per_text` to subsample instead.

`generate_dataset` returns a `GeneratedDataset` — it unpacks just like the old `(dataset, ground_truth)` tuple, but also has convenience methods (`.head()`, `.tail()`, `.sample()`, `.describe_text()`, `.text_ids_by_k()`) if you keep it as one object.

In [ ]:
result = pool.generate_dataset(
    n_texts=10,
    n_annotators_per_text=100,   # None (default) uses the full pool; capped at pool_size if larger
    noise=0.05,                  # prob. of a fully random rating (outlier noise)
    seed=42,
)

# Either unpack directly...
dataset, ground_truth = result

# ...or keep the result object and use it directly:
result.head()

## 3. Inspect the dataset with `.head()`, `.tail()`, `.sample()`

These behave exactly like their pandas equivalents, so you can eyeball the raw rows without pulling `.data` out manually.

In [ ]:
print(result.head(5))
print()
print(result.tail(3))
print()
print(result.sample(3, random_state=0))

## 4. Inspect one text's ground truth with `.describe_text()`

For an active text (`k > 0`), this prints the **lean** (which values push toxic vs civil, per active dimension), the **alpha** (intensity) drawn for each active dimension, and the actual observed rating counts. For a `k=0` text, it prints the negative-control peak/spread instead.

`.text_ids_by_k(k)` finds example text_ids for a given k, so you don't have to search the ground truth dict by hand.

In [ ]:
# One k=0 (negative control) example
k0_ids = result.text_ids_by_k(0)
if k0_ids:
    result.describe_text(k0_ids[0])
    print()

# One k=2 example
k2_ids = result.text_ids_by_k(2)
if k2_ids:
    result.describe_text(k2_ids[0])

## 5. Per-text nDFU

nDFU (Normalized Distance from Unimodality, Pavlopoulos & Likas 2023) is computed via the vectorized reference implementation in `polarimetrics.ndfu`. Shown here directly on one text's ratings, overall and split by one active dimension.

In [ ]:
from polartox.ndfu import ndfu_batch

def ndfu_for_ratings(ratings, scale=5):
    counts = np.bincount(ratings, minlength=scale + 1)[1:]
    return ndfu_batch(counts.reshape(1, -1))[0]

example_id = k2_ids[0] if k2_ids else result.text_ids_by_k(1)[0]
text_data = result.data[result.data["text_id"] == example_id]
overall = ndfu_for_ratings(text_data["rating"].to_numpy())
print(f"Overall nDFU for text {example_id}: {overall:.4f}\n")

first_active_dim = result.ground_truth[example_id]["active_dims"][0]
print(f"nDFU by '{first_active_dim}':")
for value, g in text_data.groupby(first_active_dim):
    print(f"  {value:<15} n={len(g):<4} nDFU={ndfu_for_ratings(g['rating'].to_numpy()):.4f}")

## 6. Dataset-wide nDFU summary, by depth k

At scale, generate a larger dataset and check the key validation properties: full nDFU range, a clean near-zero `k=0` negative control, and nDFU increasing with `k`.

In [ ]:
big_result = pool.generate_dataset(n_texts=2000, n_annotators_per_text=None, noise=0.05, seed=42)
big_dataset, big_ground_truth = big_result

n_texts = len(big_ground_truth)
scale = pool.scale

overall_hist = (
    big_dataset.groupby(["text_id", "rating"]).size()
    .unstack(fill_value=0)
    .reindex(columns=range(1, scale + 1), fill_value=0)
    .reindex(range(n_texts), fill_value=0)
    .to_numpy()
)
overall_ndfu = ndfu_batch(overall_hist)
k_per_text = np.array([len(big_ground_truth[t]["active_dims"]) for t in range(n_texts)])

ndfu_df = pd.DataFrame({"ndfu": overall_ndfu, "k": k_per_text})
print("Overall nDFU stats:")
print(pd.Series(overall_ndfu).describe())
print("\nnDFU by depth k:")
print(ndfu_df.groupby("k")["ndfu"].agg(["mean", "min", "max", "count"]))

## 7. Visualise — rating distribution for one active dimension

For a text with `k >= 1`, plot the rating histogram split by one active dimension's values, to see the toxic/civil separation directly.

In [ ]:
example_text_id = big_result.text_ids_by_k(1)[0] if big_result.text_ids_by_k(1) else next(
    tid for tid, cfg in big_ground_truth.items() if cfg["active_dims"]
)
cfg = big_ground_truth[example_text_id]
dim = cfg["active_dims"][0]
values = pool.dimensions[dim]

text_data = big_dataset[big_dataset["text_id"] == example_text_id]
bins = range(1, scale + 2)

fig, axes = plt.subplots(1, len(values), figsize=(3 * len(values), 3), sharey=True)
fig.suptitle(f"Text {example_text_id} (k={len(cfg['active_dims'])}) — ratings by {dim}")

for ax, value in zip(axes, values):
    subset = text_data[text_data[dim] == value]["rating"]
    ax.hist(subset, bins=bins, align="left", color="steelblue", edgecolor="white")
    ax.set_title(value)
    ax.set_xlabel("rating")
    ax.set_xticks(range(1, scale + 1))

axes[0].set_ylabel("count")
plt.tight_layout()
plt.show()